<a href="https://colab.research.google.com/github/kjahan/llm_limitations/blob/main/notebooks/question_answering_using_llms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing question answering

How can we measure the quality of an answwer for a question?

`1. USing traditional NLP methods`

`2. Using Gen AI`

In [1]:
!pip3 install --upgrade openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.6/73.6 kB 1.3 MB/s eta 0:00:00


In [2]:
import openai
import getpass

from io import StringIO
from contextlib import redirect_stdout

## API Keys

In [3]:
try:
    openai.api_key = getpass.getpass()
except Exception as error:
    print('ERROR', error)

··········


In [4]:
MODEL = "gpt-3.5-turbo"
system_promp = "You are a professional assistant who answers questions based on facts. If the question is tricky, non-sensical or incomplete respond with Unknown. Only answer a question if you are very confident."

## Biology question and answer

https://ocw.mit.edu/courses/7-013-introductory-biology-spring-2013/225d4b2aea64642edd6b3f0180a6239a_MIT7_013S13_Exam_1.pdf

https://ocw.mit.edu/courses/7-013-introductory-biology-spring-2013/cc8dca3c86361dd6a07b3e947bbb4379_MIT7_013S13_Exam_1Sol.pdf

In [5]:
question = """
Monomers for which class(s) of macromolecules always have phosphorous? Circle all that apply.
A) Carbohydrates B) Proteins C) Lipids D) DNA E) RNA
"""

correct_answer = "D) DNA E) RNA"

In [6]:
user_prompt = "Answer the following multiple choice question:\n" + question

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text = response['choices'][0]['message']['content']

In [7]:
text

'D) DNA E) RNA'

## Groundness

https://github.com/truera/trulens/blob/main/trulens_eval/trulens_eval/feedback_prompts.py

In [8]:
LLM_GROUNDEDNESS = """You are a INFORMATION OVERLAP classifier; providing the overlap of information between two statements.
Respond only as a number from 1 to 10 where 1 is no information overlap and 10 is all information is overlapping.
Never elaborate.

STATEMENT 1: {}

STATEMENT 2: {}

INFORMATION OVERLAP: """.format(correct_answer, text)

In [9]:
LLM_GROUNDEDNESS

'You are a INFORMATION OVERLAP classifier; providing the overlap of information between two statements.\nRespond only as a number from 1 to 10 where 1 is no information overlap and 10 is all information is overlapping.\nNever elaborate.\n\nSTATEMENT 1: D) DNA E) RNA\n\nSTATEMENT 2: D) DNA E) RNA\n\nINFORMATION OVERLAP: '

In [10]:
user_prompt = LLM_GROUNDEDNESS

response = openai.ChatCompletion.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": user_prompt}
    ],
    temperature=0,
)

text_2 = response['choices'][0]['message']['content']

In [11]:
text_2

'10'

## Entailement?

https://nlp.stanford.edu/projects/snli/

https://huggingface.co/roberta-large-mnli?text=D%29+DNA+E%29+RNA.+D%29+DNA+E%29+RNA

https://huggingface.co/MoritzLaurer/MiniLM-L6-mnli-binary?text=I+liked+the+movie.+%5BSEP%5D+The+movie+was+good.

In [12]:
!pip install transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 46.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 20.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 96.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.1 MB/s eta 0:00:00


In [13]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_name = "MoritzLaurer/MiniLM-L6-mnli-binary"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
# Just right before the actual usage
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [14]:
premise = "I liked the movie"
hypothesis = "The movie was good."

input = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt")

output = model(input["input_ids"].to(device))  # device = "cuda:0" or "cpu"
prediction = torch.softmax(output["logits"][0], -1).tolist()
label_names = ["entailment", "not_entailment"]
prediction = {name: round(float(pred) * 100, 1) for pred, name in zip(prediction, label_names)}
print(prediction)

{'entailment': 68.2, 'not_entailment': 31.8}


## Can we put two answers and see if they agree or not?



In [15]:
premise = correct_answer
hypothesis = "D) DNA E) RNA"

input = tokenizer(premise, hypothesis, truncation=True, return_tensors="pt")

output = model(input["input_ids"].to(device))  # device = "cuda:0" or "cpu"
prediction = torch.softmax(output["logits"][0], -1).tolist()
label_names = ["entailment", "not_entailment"]
prediction = {name: round(float(pred) * 100, 1) for pred, name in zip(prediction, label_names)}
print(prediction)

{'entailment': 47.5, 'not_entailment': 52.5}


In [16]:
premise

'D) DNA E) RNA'

## Entailements?

https://huggingface.co/roberta-large-mnli?text=D%29+DNA+E%29+RNA.+D%29+DNA+E%29+RNA

In [17]:
from transformers import pipeline
classifier = pipeline('zero-shot-classification', model='roberta-large-mnli')


Some weights of the model checkpoint at roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [18]:
sequence_to_classify = "The CEO had a strong handshake."
candidate_labels = ['male', 'female']
hypothesis_template = "This text speaks about a {} profession."
classifier(sequence_to_classify, candidate_labels, hypothesis_template=hypothesis_template)


{'sequence': 'The CEO had a strong handshake.',
 'labels': ['male', 'female'],
 'scores': [0.8384836316108704, 0.16151636838912964]}

In [21]:
sequence_to_classify = "D) DNA E) RNA"
candidate_labels = ['D) DNA E) RNA', 'B) Proteins']
hypothesis_template = "This text support {}."
classifier(sequence_to_classify, candidate_labels, hypothesis_template=hypothesis_template)

{'sequence': 'D) DNA E) RNA',
 'labels': ['D) DNA E) RNA', 'B) Proteins'],
 'scores': [0.9357563853263855, 0.06424365937709808]}